# Handwritten Digit Recognition with Nearest Neighbors

A from-scratch k-nearest-neighbor classifier for MNIST digits, written in NumPy.
No training step: a new image gets the label of the most similar images in the training set.

* 7,500 training images, 1,000 test images (balanced, 28×28 grayscale)
* **95.4% test accuracy** with plain 1-NN
* Fully vectorized distance computation: the whole test set is classified in one matrix product
* Compared against scikit-learn's KD-tree and ball-tree
* Browser demo where you can draw your own digits (last cell opens it)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

train_x = np.load('MNIST/train_data.npy')    # (7500, 784), pixel values 0-255
train_y = np.load('MNIST/train_labels.npy')
test_x = np.load('MNIST/test_data.npy')      # (1000, 784)
test_y = np.load('MNIST/test_labels.npy')

print('train:', train_x.shape, ' test:', test_x.shape)
print('images per digit (train):', np.bincount(train_y))

## The data
Each image is stored as a flat 784-vector; reshaping to 28×28 recovers the picture.

In [ ]:
def show_grid(images, titles, cols=10):
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.1, rows * 1.3))
    for ax in np.ravel(axes):
        ax.axis('off')
    for ax, img, title in zip(np.ravel(axes), images, titles):
        ax.imshow(img.reshape(28, 28), cmap='gray_r')
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()

# first two examples of every digit
idx = np.concatenate([np.flatnonzero(train_y == d)[:2] for d in range(10)])
show_grid(train_x[idx], train_y[idx])

## Vectorized nearest neighbors

Distance is squared Euclidean distance between pixel vectors. Dropping the square root doesn't change which neighbor is closest.

Looping over 7,500 training images for each test image is slow in Python. Expanding the square turns it into matrix math:

$$\|x - y\|^2 = \|x\|^2 - 2\,x \cdot y + \|y\|^2$$

So the full test × train distance matrix is one matrix product plus two broadcasts.

In [ ]:
def pairwise_sq_dists(a, b):
    a, b = a.astype(np.float64), b.astype(np.float64)
    d = (a**2).sum(1)[:, None] - 2 * a @ b.T + (b**2).sum(1)[None, :]
    return np.maximum(d, 0)  # clip tiny negatives from floating-point error

def knn_predict(x, k=1):
    d = pairwise_sq_dists(x, train_x)
    nearest = np.argsort(d, axis=1)[:, :k]
    votes = train_y[nearest]
    # majority vote; ties go to the smallest label
    return np.array([np.bincount(v, minlength=10).argmax() for v in votes])

t = time.time()
pred = knn_predict(test_x, k=1)
print(f'1-NN test accuracy: {np.mean(pred == test_y):.1%}  ({time.time() - t:.2f}s for 1,000 images)')

## Comparison with tree-based search
KD-trees and ball-trees trade a one-time build cost for faster queries. In 784 dimensions they don't prune much, so the brute-force matrix product holds its own.

In [ ]:
from sklearn.neighbors import BallTree, KDTree

for Tree in (KDTree, BallTree):
    t0 = time.time(); tree = Tree(train_x); t1 = time.time()
    nn = tree.query(test_x, k=1, return_distance=False).ravel(); t2 = time.time()
    same = np.array_equal(train_y[nn], pred)
    print(f'{Tree.__name__:8s} build {t1 - t0:.2f}s  query {t2 - t1:.2f}s  matches NumPy: {same}')

## Choosing k
Voting over several neighbors is the usual way to smooth out noisy training examples. On this dataset it doesn't help: plain 1-NN scores best, which is why the demo uses it.

In [ ]:
ks = range(1, 11)
dists = pairwise_sq_dists(test_x, train_x)
order = np.argsort(dists, axis=1)
accs = [np.mean([np.bincount(v, minlength=10).argmax() for v in train_y[order[:, :k]]] == test_y) for k in ks]

plt.plot(ks, accs, 'o-')
plt.xlabel('k'); plt.ylabel('test accuracy'); plt.xticks(ks); plt.grid(alpha=.3)
plt.show()
print({k: f'{a:.1%}' for k, a in zip(ks, accs)})

## Exploring Error

In [ ]:
cm = np.zeros((10, 10), dtype=int)
np.add.at(cm, (test_y, pred), 1)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
plt.xlabel('predicted'); plt.ylabel('true'); plt.title('1-NN confusion matrix')
plt.xticks(range(10)); plt.yticks(range(10)); plt.colorbar()
plt.show()

Each misclassified test image next to the training image it matched. Most errors are genuinely ambiguous strokes or digits drawn at an unusual slant.

In [ ]:
wrong = np.flatnonzero(pred != test_y)[:8]
nn_idx = order[wrong, 0]
imgs = np.stack([test_x[wrong], train_x[nn_idx]], axis=1).reshape(-1, 784)
titles = [t for w, n in zip(wrong, nn_idx) for t in (f'test: {test_y[w]}', f'match: {train_y[n]}')]
show_grid(imgs, titles, cols=8)

## The average digit
Mean of all training images per class — a blurry prototype of each digit.

In [ ]:
means = np.stack([train_x[train_y == d].mean(0) for d in range(10)])
show_grid(means, range(10))

## Export for the browser demo
Packs the training set (`uint8` pixels followed by labels) into a base64 string inside `docs/train.js`. `docs/index.html` loads it with a plain `<script>` tag, so the demo works when opened straight from disk, with no server needed, and runs the same 1-NN search in JavaScript.

In [ ]:
import os, base64
os.makedirs('docs', exist_ok=True)
raw = train_x.astype(np.uint8).tobytes() + train_y.astype(np.uint8).tobytes()
with open('docs/train.js', 'w') as f:
    f.write('const TRAIN_B64 = "' + base64.b64encode(raw).decode() + '";\n')
print(os.path.getsize('docs/train.js') / 1e6, 'MB')

## Open the demo
Opens `docs/index.html` in your default browser.

In [ ]:
import webbrowser, pathlib
webbrowser.open(pathlib.Path('docs/index.html').resolve().as_uri())